In [ ]:
# CELL 1: SETUP & SEM GENERATOR
import argparse
import csv
import json
import math
import os
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
from scipy.ndimage import map_coordinates, gaussian_filter1d
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
EVAL_DIR = WORKDIR / "eval_pairs"
OUTPUT_DIR = WORKDIR / "eval_outputs"
WEIGHTS_DIR = WORKDIR / "weights"
for d in [EVAL_DIR, OUTPUT_DIR, WEIGHTS_DIR]: d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Active Compute Device: {DEVICE}")

# Physical Parameters
PITCH_X_NM, PITCH_Y_NM = 170.0, 125.0        
LINE_WIDTH_X_NM, LINE_WIDTH_Y_NM = 36.0, 46.0    
CONTACT_R_NM, EDGE_SOFTNESS_NM = 28.0, 1.5    
SUBGRID_PITCH_NM, SUBGRID_LINE_WIDTH_NM = 220.0, 6.0
SUBGRID_VAL, SUBGRID_SOFTNESS_NM = 105.0, 2.0 
TRENCH_VAL, BG_VAL, LINE_VAL, CONTACT_VAL, STREET_VAL = 28.0, 65.0, 158.0, 245.0, 84.0         

MAT_SIZE_NM, SEPARATOR_WIDTH_NM = 2600.0, 320.0
P_STRADDLE = 0.35
BEAM_SPOT_SIGMA_NM = 5.0

REF_PX_SIZE_NM, REF_SIZE_PX = 1.0, 1000
REF_FOV_NM = REF_SIZE_PX * REF_PX_SIZE_NM
SEARCH_PX_SIZE_NM, SEARCH_SIZE_PX = 10.0, 1000
SEARCH_FOV_NM = SEARCH_SIZE_PX * SEARCH_PX_SIZE_NM

REFERENCE_DOSE, SEARCH_DOSE = 2000.0, 200.0
SEARCH_RASTER_SHEAR_PX, SEARCH_ROW_JITTER_PX = 1.5, 0.5
BARREL_DISTORTION_K1 = -0.05  
ASTIGMATISM_RANGE = (0.85, 1.15)

SUPERSAMPLE = 4
SEARCH_NATIVE_PX_NM = SEARCH_PX_SIZE_NM / SUPERSAMPLE
CANVAS_FOV_NM = 30000.0
CANVAS_PX_NM = SEARCH_NATIVE_PX_NM
CANVAS_PX = int(round(CANVAS_FOV_NM / CANVAS_PX_NM))

CONTACT_ROTATION_DEG_RANGE = (-3.0, 3.0)
CONTACT_SCALE_JITTER_RANGE = (-0.15, 0.15)

_cache = {}

def sigmoid(x): return 1.0 / (1.0 + np.exp(-np.clip(x, -60, 60)))

def get_contact_jitter_grid(jitter_seed=0):
    key = ("contact_jitter", jitter_seed)
    if key not in _cache:
        nx, ny = int(np.ceil(CANVAS_FOV_NM / (PITCH_X_NM * 0.5))) + 2, int(np.ceil(CANVAS_FOV_NM / (PITCH_Y_NM * 0.5))) + 2
        rng = np.random.default_rng(jitter_seed)
        _cache[key] = (rng.uniform(*CONTACT_ROTATION_DEG_RANGE, size=(ny, nx)).astype(np.float32),
                       1.0 + rng.uniform(*CONTACT_SCALE_JITTER_RANGE, size=(ny, nx)).astype(np.float32))
    return _cache[key]

def get_mat_pitch_grid(jitter_seed=0):
    key = ("mat_pitch", jitter_seed)
    if key not in _cache:
        nm = int(np.ceil(CANVAS_FOV_NM / MAT_SIZE_NM)) + 2
        _cache[key] = np.random.default_rng(jitter_seed + 101).choice(np.array([0.65, 0.85, 1.0, 1.0, 1.35, 1.6], dtype=np.float32), size=(nm, nm))
    return _cache[key]

def layout_intensity(x_nm, y_nm, jitter_seed=0):
    x_nm, y_nm = x_nm.astype(np.float32, copy=False), y_nm.astype(np.float32, copy=False)

    sub_dx = np.mod(x_nm + SUBGRID_PITCH_NM / 2, SUBGRID_PITCH_NM) - SUBGRID_PITCH_NM / 2
    sub_dy = np.mod(y_nm + SUBGRID_PITCH_NM / 2, SUBGRID_PITCH_NM) - SUBGRID_PITCH_NM / 2
    subgrid_resp = np.maximum(sigmoid((SUBGRID_LINE_WIDTH_NM / 2 - np.abs(sub_dx)) / SUBGRID_SOFTNESS_NM),
                              sigmoid((SUBGRID_LINE_WIDTH_NM / 2 - np.abs(sub_dy)) / SUBGRID_SOFTNESS_NM))

    pitch_grid = get_mat_pitch_grid(jitter_seed)
    m_col = np.clip((x_nm / MAT_SIZE_NM).astype(np.int32), 0, pitch_grid.shape[1] - 1)
    m_row = np.clip((y_nm / MAT_SIZE_NM).astype(np.int32), 0, pitch_grid.shape[0] - 1)
    l_ps = pitch_grid[m_row, m_col]

    l_px, l_py = PITCH_X_NM * l_ps, PITCH_Y_NM * l_ps
    l_lw_x, l_lw_y, l_cr = LINE_WIDTH_X_NM * np.sqrt(l_ps), LINE_WIDTH_Y_NM * np.sqrt(l_ps), CONTACT_R_NM * np.sqrt(l_ps)

    dx = np.mod(x_nm + l_px / 2, l_px) - l_px / 2
    dy = np.mod(y_nm + l_py / 2, l_py) - l_py / 2
    line_mask = np.maximum(sigmoid((l_lw_x / 2 - np.abs(dx)) / EDGE_SOFTNESS_NM), sigmoid((l_lw_y / 2 - np.abs(dy)) / EDGE_SOFTNESS_NM))
    trench_resp = (1.0 - sigmoid((l_lw_x / 2 - np.abs(dx)) / EDGE_SOFTNESS_NM)) * (1.0 - sigmoid((l_lw_y / 2 - np.abs(dy)) / EDGE_SOFTNESS_NM))

    core_img = (BG_VAL * (1.0 - trench_resp) + TRENCH_VAL * trench_resp) + (LINE_VAL - BG_VAL) * line_mask

    cell_col, cell_row = (x_nm / l_px).astype(np.int32), (y_nm / l_py).astype(np.int32)
    stagger_mask = ((cell_col + cell_row) % 2 == 0).astype(np.float32)

    rot_grid, scale_grid = get_contact_jitter_grid(jitter_seed)
    j_col, j_row = np.clip(cell_col, 0, rot_grid.shape[1] - 1), np.clip(cell_row, 0, rot_grid.shape[0] - 1)
    
    theta = np.radians(-rot_grid[j_row, j_col]).astype(np.float32)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    
    cell_scale = scale_grid[j_row, j_col]
    r = np.sqrt(((dx * cos_t - dy * sin_t) / cell_scale) ** 2 + ((dx * sin_t + dy * cos_t) / cell_scale) ** 2)
    contact_resp = sigmoid((l_cr - r) / EDGE_SOFTNESS_NM) * stagger_mask

    core_img = core_img * (1.0 - contact_resp) + CONTACT_VAL * contact_resp
    core_edge_dist = np.minimum(np.abs(np.abs(dx) - l_lw_x / 2), np.abs(np.abs(dy) - l_lw_y / 2))

    ddx = np.mod(x_nm + MAT_SIZE_NM / 2, MAT_SIZE_NM) - MAT_SIZE_NM / 2
    ddy = np.mod(y_nm + MAT_SIZE_NM / 2, MAT_SIZE_NM) - MAT_SIZE_NM / 2
    street_resp = np.maximum(sigmoid((SEPARATOR_WIDTH_NM / 2 - np.abs(ddx)) / EDGE_SOFTNESS_NM), sigmoid((SEPARATOR_WIDTH_NM / 2 - np.abs(ddy)) / EDGE_SOFTNESS_NM))

    return core_img * (1.0 - street_resp) + (STREET_VAL + (SUBGRID_VAL - STREET_VAL) * subgrid_resp) * street_resp, core_edge_dist * (1.0 - street_resp) + (SEPARATOR_WIDTH_NM / 2) * street_resp

def build_canvas(jitter_seed=0):
    cache_key = ("canvas", jitter_seed)
    if cache_key in _cache: return _cache[cache_key], _cache[("canvas_edge", jitter_seed)]

    coords = ((np.arange(CANVAS_PX) + 0.5) * CANVAS_PX_NM).astype(np.float32)
    img_full, edge_full = np.empty((CANVAS_PX, CANVAS_PX), dtype=np.float32), np.empty((CANVAS_PX, CANVAS_PX), dtype=np.float32)

    for r0 in range(0, CANVAS_PX, 400):
        r1 = min(r0 + 400, CANVAS_PX)
        xs, ys = np.meshgrid(coords, coords[r0:r1])
        img_full[r0:r1, :], edge_full[r0:r1, :] = layout_intensity(xs, ys, jitter_seed=jitter_seed)

    rng = np.random.default_rng(jitter_seed + 555)
    astig_ratio = rng.uniform(*ASTIGMATISM_RANGE)
    b_sigma = BEAM_SPOT_SIGMA_NM / CANVAS_PX_NM
    img_full_cv = cv2.GaussianBlur(np.clip(img_full, 0, 255).astype(np.float32), (0, 0), sigmaX=b_sigma * astig_ratio, sigmaY=b_sigma / astig_ratio)
    
    _cache[cache_key] = img_full_cv
    _cache[("canvas_edge", jitter_seed)] = edge_full
    _cache[("canvas_pil", jitter_seed)] = Image.fromarray(np.clip(img_full_cv, 0, 255).astype(np.uint8), mode="L")
    _cache[("canvas_edge_pil", jitter_seed)] = Image.fromarray(edge_full.astype(np.float32), mode="F")

    return img_full_cv, edge_full

def crop_from_canvas(center_nm, fov_nm, out_px, jitter_seed=0):
    build_canvas(jitter_seed=jitter_seed)
    pil_full, pil_edge = _cache[("canvas_pil", jitter_seed)], _cache[("canvas_edge_pil", jitter_seed)]
    cx, cy = center_nm
    x0, y0 = (cx - fov_nm / 2) / CANVAS_PX_NM, (cy - fov_nm / 2) / CANVAS_PX_NM
    x1, y1 = (cx + fov_nm / 2) / CANVAS_PX_NM, (cy + fov_nm / 2) / CANVAS_PX_NM
    
    resample_filter = Image.Resampling.BOX if (x1 - x0) > out_px else Image.Resampling.LANCZOS
    return np.asarray(pil_full.resize((out_px, out_px), resample_filter, box=(x0, y0, x1, y1)), dtype=np.float32), np.asarray(pil_edge.resize((out_px, out_px), Image.Resampling.BILINEAR, box=(x0, y0, x1, y1)), dtype=np.float32)

def apply_radial_distortion(img, k1):
    h, w = img.shape
    y, x = np.mgrid[0:h, 0:w]
    x_n, y_n = (x - w / 2) / (w / 2), (y - h / 2) / (h / 2)
    r2 = x_n**2 + y_n**2
    map_x = ((x_n * (1 + k1 * r2)) * (w / 2) + (w / 2)).astype(np.float32)
    map_y = ((y_n * (1 + k1 * r2)) * (h / 2) + (h / 2)).astype(np.float32)
    return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

def apply_search_raster_artifacts(img, rng, fill_value):
    h, w = img.shape
    total_shift = (SEARCH_RASTER_SHEAR_PX * (np.arange(h) / h - 0.5) + gaussian_filter1d(rng.normal(0, SEARCH_ROW_JITTER_PX * 1.5, size=h), sigma=4.0)).astype(np.float32)
    yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    shifted = map_coordinates(img.astype(np.float32), np.stack([yy.astype(np.float32), xx - total_shift[:, None]]), order=1, mode="constant", cval=fill_value)
    return apply_radial_distortion(shifted, k1=BARREL_DISTORTION_K1 * rng.choice([-1.0, 1.0]))

def generate_pair(seed, boundary_bias=0.35, out_dir=None, jitter_seed=0):
    rng = np.random.default_rng(seed)
    
    drift_max = 800.0  
    margin = SEARCH_FOV_NM / 2 + drift_max
    valid_lo, valid_hi = margin, CANVAS_FOV_NM - margin
    is_boundary_case = rng.random() < boundary_bias
    
    if is_boundary_case:
        axis = rng.integers(0, 2)
        boundary = rng.integers(int(np.ceil(valid_lo / MAT_SIZE_NM)), int(np.floor(valid_hi / MAT_SIZE_NM)) + 1) * MAT_SIZE_NM
        coord_on_axis = float(np.clip(boundary + rng.uniform(-REF_FOV_NM * 0.15, REF_FOV_NM * 0.15), valid_lo, valid_hi))
        other = float(rng.uniform(valid_lo, valid_hi))
        ref_center = (coord_on_axis, other) if axis == 0 else (other, coord_on_axis)
    else:
        for _ in range(25):
            cx, cy = float(rng.uniform(valid_lo, valid_hi)), float(rng.uniform(valid_lo, valid_hi))
            if min(cx % MAT_SIZE_NM, MAT_SIZE_NM - (cx % MAT_SIZE_NM)) > REF_FOV_NM * 0.6 and min(cy % MAT_SIZE_NM, MAT_SIZE_NM - (cy % MAT_SIZE_NM)) > REF_FOV_NM * 0.6:
                ref_center = (cx, cy)
                break
        else: ref_center = (cx, cy)
            
    drift = rng.uniform(-drift_max, drift_max, size=2)
    search_center = (ref_center[0] + drift[0], ref_center[1] + drift[1])

    ref_img, ref_edge = crop_from_canvas(ref_center, REF_FOV_NM, REF_SIZE_PX, jitter_seed=jitter_seed)
    search_img, search_edge = crop_from_canvas(search_center, SEARCH_FOV_NM, SEARCH_SIZE_PX, jitter_seed=jitter_seed)

    ref_img = np.clip(ref_img + 18 * np.exp(-(np.clip(ref_edge, 0, None) ** 2) / (2 * 1.5**2)), 0, 255)
    search_img = apply_search_raster_artifacts(np.clip(search_img + 13 * np.exp(-(np.clip(search_edge, 0, None) ** 2) / (2 * 1.0**2)), 0, 255), rng, fill_value=STREET_VAL)

    for img, strength, bg_val, blobs in [(ref_img, 8, ref_img < (BG_VAL + LINE_VAL) / 2, 3), (search_img, 8, search_img < (BG_VAL + LINE_VAL) / 2, 4)]:
        charge = np.zeros(img.shape, dtype=np.float32)
        ys, xs = np.mgrid[0:img.shape[0], 0:img.shape[1]]
        for _ in range(blobs):
            cx, cy, sigma = rng.uniform(0, img.shape[1]), rng.uniform(0, img.shape[0]), 70 * rng.uniform(0.7, 1.3)
            charge += rng.choice([-1.0, 1.0]) * strength * np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2 * sigma**2))
        img[:] = np.clip(img + charge * bg_val, 0, 255)

    ref_noisy = np.clip(rng.poisson(np.clip(ref_img, 1, 255) * (REFERENCE_DOSE/255.0)) / (REFERENCE_DOSE/255.0) + rng.normal(0, 2.5, size=ref_img.shape), 0, 255).astype(np.uint8)
    search_noisy = np.clip(rng.poisson(np.clip(search_img, 1, 255) * (SEARCH_DOSE/255.0)) / (SEARCH_DOSE/255.0) + rng.normal(0, 2.5, size=search_img.shape), 0, 255).astype(np.uint8)

    gt_w = REF_FOV_NM / SEARCH_PX_SIZE_NM
    gt_x0 = ((ref_center[0] - REF_FOV_NM / 2) - (search_center[0] - SEARCH_FOV_NM / 2)) / SEARCH_PX_SIZE_NM
    gt_y0 = ((ref_center[1] - REF_FOV_NM / 2) - (search_center[1] - SEARCH_FOV_NM / 2)) / SEARCH_PX_SIZE_NM

    meta = {"sample": out_dir.name if out_dir else f"seed_{seed}", "seed": seed, "is_boundary_case": is_boundary_case,
            "gt_cx": float(gt_x0 + gt_w / 2), "gt_cy": float(gt_y0 + gt_w / 2)}

    if out_dir is not None:
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(ref_noisy, mode="L").save(out_dir / "reference.png")
        Image.fromarray(search_noisy, mode="L").save(out_dir / "search.png")
        with open(out_dir / "meta.json", "w") as f: json.dump(meta, f, indent=2)

    return ref_noisy, search_noisy, meta

[*] Active Compute Device: cuda


In [ ]:
# CELL 2: NEURAL NETWORK ARCHITECTURE & SPATIAL CENTER PRIOR

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.SiLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
        )
        self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride, bias=False), nn.BatchNorm2d(out_c)) if stride != 1 or in_c != out_c else nn.Sequential()

    def forward(self, x): return F.silu(self.conv(x) + self.shortcut(x))

class FeatureBackbone(nn.Module):
    def __init__(self, in_channels=1, base_channels=32):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_channels, base_channels, 3, 2, 1, bias=False), nn.BatchNorm2d(base_channels), nn.SiLU(inplace=True))
        self.layer1 = ConvBlock(base_channels, base_channels * 2, stride=2)
        self.layer2 = ConvBlock(base_channels * 2, base_channels * 4, stride=2)
        self.layer3 = ConvBlock(base_channels * 4, base_channels * 4, stride=1)

    def forward(self, x): return self.layer3(self.layer2(self.layer1(self.stem(x))))

class DepthwiseXCorr(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv_kernel = nn.Sequential(nn.Conv2d(channels, channels, 3, 1, 1, bias=False), nn.BatchNorm2d(channels), nn.SiLU(inplace=True))
        self.conv_search = nn.Sequential(nn.Conv2d(channels, channels, 3, 1, 1, bias=False), nn.BatchNorm2d(channels), nn.SiLU(inplace=True))

    def forward(self, z_feat, x_feat):
        z_feat, x_feat = self.conv_kernel(z_feat), self.conv_search(x_feat)
        out = []
        for i in range(x_feat.shape[0]):
            k, x_pad = z_feat[i:i+1], F.pad(x_feat[i:i+1], (z_feat.shape[3]//2, z_feat.shape[3]//2, z_feat.shape[2]//2, z_feat.shape[2]//2), mode="reflect")
            out.append(F.conv2d(x_pad, k.permute(1, 0, 2, 3), groups=x_feat.shape[1]))
        return torch.cat(out, dim=0)

class DriftSenseNet(nn.Module):
    def __init__(self, base_c=32):
        super().__init__()
        self.backbone = FeatureBackbone(in_channels=1, base_channels=base_c)
        feat_c = base_c * 4
        self.xcorr = DepthwiseXCorr(feat_c)

        self.heatmap_head = nn.Sequential(nn.Conv2d(feat_c, feat_c, 3, 1, 1), nn.BatchNorm2d(feat_c), nn.SiLU(inplace=True), nn.Conv2d(feat_c, 1, 1))
        self.offset_head = nn.Sequential(nn.Conv2d(feat_c, feat_c, 3, 1, 1), nn.BatchNorm2d(feat_c), nn.SiLU(inplace=True), nn.Conv2d(feat_c, 2, 1))
        
        # The Spatial Center Prior. 
        # A static 2D quadratic penalty mapping directly over the 125x125 heatmap grid.
        # It adds log(Gaussian) to the logits, penalizing edges natively during training.
        ys, xs = np.mgrid[0:125, 0:125]
        log_prior = -((xs - 62.5)**2 + (ys - 62.5)**2) / (2.0 * (40.0**2))
        self.register_buffer('log_prior', torch.from_numpy(log_prior).float().unsqueeze(0).unsqueeze(0))

    def forward(self, template, search):
        corr_feat = self.xcorr(self.backbone(template), self.backbone(search))
        
        heatmap_logits = self.heatmap_head(corr_feat) + self.log_prior
        offset = self.offset_head(corr_feat)
        return heatmap_logits, offset

In [ ]:
# CELL 3: DATASTREAM & TRAINING ENGINE

STRIDE = 8

class SpatialInfoNCELoss(nn.Module):
    """
    Contrastive Loss: Pushes the true location probability to 1.0 while 
    simultaneously squashing identical periodic clones to 0.0.
    """
    def __init__(self, temperature=0.2): # Softened to 0.2 to prevent gradient thrashing
        super().__init__()
        self.temperature = temperature
        
    def forward(self, pred_hm_logits, int_coords):
        B, C, H, W = pred_hm_logits.shape
        pred_flat = pred_hm_logits.view(B, H * W) / self.temperature
        target_idx = int_coords[:, 1] * W + int_coords[:, 0]
        return F.cross_entropy(pred_flat, target_idx)

class StreamingSEMDataset(Dataset):
    def __init__(self, num_samples=960, is_train=True):
        self.num_samples, self.is_train = num_samples, is_train

    def __len__(self): return self.num_samples

    def __getitem__(self, idx):
        ref_u8, search_u8, meta = generate_pair(seed=(idx * 17 + 101) if self.is_train else (idx + 9000), boundary_bias=0.50 if self.is_train else 0.40)

        ref_small = np.asarray(Image.fromarray(ref_u8).filter(ImageFilter.GaussianBlur(radius=3.33)).resize((100, 100), Image.Resampling.LANCZOS), dtype=np.float32)
        gt_x, gt_y = meta["gt_cx"], meta["gt_cy"]
        int_x, int_y = int(np.clip(math.floor(gt_x / STRIDE), 0, 124)), int(np.clip(math.floor(gt_y / STRIDE), 0, 124))

        return {
            "template": torch.from_numpy(ref_small / 255.0).unsqueeze(0).float(),
            "search": torch.from_numpy(search_u8.astype(np.float32) / 255.0).unsqueeze(0).float(),
            "int_coord": torch.tensor([int_x, int_y], dtype=torch.long),
            "offset_gt": torch.tensor([gt_x / STRIDE - int_x, gt_y / STRIDE - int_y], dtype=torch.float32),
            "raw_gt": torch.tensor([gt_x, gt_y], dtype=torch.float32),
        }

def train_model(epochs=10, batch_size=8, lr=3e-4):
    train_loader = DataLoader(StreamingSEMDataset(num_samples=960, is_train=True), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(StreamingSEMDataset(num_samples=160, is_train=False), batch_size=batch_size, shuffle=False)

    model = DriftSenseNet(base_c=32).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    spatial_nce = SpatialInfoNCELoss(temperature=0.2)
    smooth_l1 = nn.SmoothL1Loss(reduction="mean")

    best_px_error = float("inf")
    print(f"\n[*] Starting InfoNCE Training Stream on {DEVICE} ({epochs} Epochs)")

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, t0 = 0.0, time.time()

        for batch in train_loader:
            tmpl, srch, off_gt, coords = batch["template"].to(DEVICE), batch["search"].to(DEVICE), batch["offset_gt"].to(DEVICE), batch["int_coord"].to(DEVICE)
            
            optimizer.zero_grad()
            pred_hm_logits, pred_off = model(tmpl, srch)

            loss = spatial_nce(pred_hm_logits, coords) + 2.0 * smooth_l1(pred_off[torch.arange(tmpl.size(0)), :, coords[:, 1], coords[:, 0]], off_gt)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()

        model.eval()
        errors = []
        with torch.no_grad():
            for batch in val_loader:
                tmpl, srch, raw_gt = batch["template"].to(DEVICE), batch["search"].to(DEVICE), batch["raw_gt"].numpy()
                pred_hm_logits, pred_off = model(tmpl, srch)

                for i in range(tmpl.size(0)):
                    hm, off = torch.sigmoid(pred_hm_logits[i, 0]).cpu().numpy(), pred_off[i].cpu().numpy()
                    y_max, x_max = np.unravel_index(np.argmax(hm), hm.shape)
                    errors.append(np.hypot((x_max + off[0, y_max, x_max]) * STRIDE - raw_gt[i, 0], (y_max + off[1, y_max, x_max]) * STRIDE - raw_gt[i, 1]))

        mean_err, pass_1px, pass_2px = float(np.mean(errors)), (np.array(errors) <= 1.0).mean() * 100.0, (np.array(errors) <= 2.0).mean() * 100.0
        print(f"Epoch [{ep:02d}/{epochs:02d}] ({time.time()-t0:.1f}s) | Loss: {total_loss/len(train_loader):.4f} | Mean Err: {mean_err:.2f}px | <=1px: {pass_1px:.1f}% | <=2px: {pass_2px:.1f}%")

        if mean_err < best_px_error:
            best_px_error = mean_err
            torch.save(model.state_dict(), WEIGHTS_DIR / "driftsense_best.pth")

    print(f"[+] Model checkpoint exported: '{WEIGHTS_DIR / 'driftsense_best.pth'}'")
    return model

In [ ]:
# CELL 4: BENCHMARK EVALUATOR & EXECUTION

def predict_pair(model, ref_input, search_input):
    t0 = time.perf_counter()
    ref_img = cv2.imread(str(ref_input), cv2.IMREAD_GRAYSCALE) if isinstance(ref_input, (str, Path)) else ref_input
    search_img = cv2.imread(str(search_input), cv2.IMREAD_GRAYSCALE) if isinstance(search_input, (str, Path)) else search_input

    ref_small = np.asarray(Image.fromarray(ref_img).filter(ImageFilter.GaussianBlur(radius=3.33)).resize((100, 100), Image.Resampling.LANCZOS), dtype=np.float32)

    with torch.no_grad():
        pred_hm_logits, pred_off = model(torch.from_numpy(ref_small / 255.0).unsqueeze(0).unsqueeze(0).float().to(DEVICE), 
                                         torch.from_numpy(search_img.astype(np.float32) / 255.0).unsqueeze(0).unsqueeze(0).float().to(DEVICE))
        hm, off = torch.sigmoid(pred_hm_logits[0, 0]).cpu().numpy(), pred_off[0].cpu().numpy()

    y_idxs, x_idxs = np.where(hm >= (hm.max() * 0.90))
    best_x, best_y, min_dist = None, None, float("inf")

    for y_c, x_c in zip(y_idxs, x_idxs):
        dist = np.hypot(x_c - 500.0 / STRIDE, y_c - 500.0 / STRIDE)
        if dist < min_dist:
            min_dist, best_x, best_y = dist, (x_c + off[0, y_c, x_c]) * STRIDE, (y_c + off[1, y_c, x_c]) * STRIDE

    return float(best_x), float(best_y), (time.perf_counter() - t0) * 1000.0, hm

def run_benchmark_and_slide6(model):
    df = pd.read_csv(EVAL_DIR / "manifest.csv")
    tolerances, records = [1.0, 2.0, 3.0, 4.0, 5.0], []

    print(f"\n{'-'*75}\n[*] Benchmarking Model on {len(df)} Fixed Evaluation Pairs\n{'-'*75}")
    for _, row in df.iterrows():
        pred_x, pred_y, infer_ms, _ = predict_pair(model, EVAL_DIR / row["reference_path"], EVAL_DIR / row["search_path"])
        err = np.hypot(pred_x - row["gt_cx"], pred_y - row["gt_cy"])

        entry = {"sample": row["sample"], "is_boundary_case": row["is_boundary_case"], "gt_cx": row["gt_cx"], "gt_cy": row["gt_cy"], "pred_x": round(pred_x, 2), "pred_y": round(pred_y, 2), "error_px": round(err, 3), "time_ms": round(infer_ms, 2)}
        for tol in tolerances: entry[f"pass_{int(tol)}px"] = 1 if err <= tol else 0
        records.append(entry)

    res_df = pd.DataFrame(records)
    res_df.to_csv(OUTPUT_DIR / "benchmark_results.csv", index=False)

    print(f"\n{'='*65}\n               ACCURACY REPORT CARD                    \n{'='*65}")
    print(f"Total Evaluated Test Cases : {len(res_df)}\nMean Inference Speed       : {res_df['time_ms'].mean():.2f} ms / pair\nMedian Localization Error  : {res_df['error_px'].median():.3f} px\nMean Localization Error    : {res_df['error_px'].mean():.3f} px\n")
    for tol in tolerances:
        passed = int(res_df[f"pass_{int(tol)}px"].sum())
        print(f"<= {int(tol)} px ({int(tol)*10:02d} nm)   | {passed:<8} | {len(res_df)-passed:<8} | {(passed / len(res_df)) * 100.0:.2f}%")
    print("=" * 65)

if __name__ == "__main__":
    rows = []
    print(f"[*] Synthesizing 30 fixed evaluation pairs into '{EVAL_DIR}'")
    for i in range(30):
        sample_name = f"sample_{i:04d}"
        _, _, meta = generate_pair(seed=42 + i, boundary_bias=1.0 if (i % 2 == 0) else 0.0, out_dir=EVAL_DIR / "pairs" / sample_name)
        rows.append({"sample": sample_name, "seed": 42 + i, "is_boundary_case": meta["is_boundary_case"], "gt_cx": meta["gt_cx"], "gt_cy": meta["gt_cy"], "reference_path": f"pairs/{sample_name}/reference.png", "search_path": f"pairs/{sample_name}/search.png"})

    with open(EVAL_DIR / "manifest.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader(); writer.writerows(rows)

    trained_model = train_model(epochs=10, batch_size=8, lr=3e-4)
    run_benchmark_and_slide6(trained_model)

[*] Synthesizing 30 fixed evaluation pairs into '/kaggle/working/eval_pairs'...


/tmp/ipykernel_23/3948553342.py:149: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  _cache[("canvas_pil", jitter_seed)] = Image.fromarray(np.clip(img_full_cv, 0, 255).astype(np.uint8), mode="L")
/tmp/ipykernel_23/3948553342.py:150: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  _cache[("canvas_edge_pil", jitter_seed)] = Image.fromarray(edge_full.astype(np.float32), mode="F")
/tmp/ipykernel_23/3948553342.py:232: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(ref_noisy, mode="L").save(out_dir / "reference.png")
/tmp/ipykernel_23/3948553342.py:233: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(search_noisy, mode="L").save(out_dir / "search.png")



[*] Starting InfoNCE Training Stream on cuda (10 Epochs)...
Epoch [01/10] (583.4s) | Loss: 3.6009 | Mean Err: 45.40px | <=1px: 5.0% | <=2px: 10.0%
Epoch [02/10] (576.8s) | Loss: 1.7784 | Mean Err: 19.93px | <=1px: 15.6% | <=2px: 46.2%
Epoch [03/10] (582.7s) | Loss: 0.6369 | Mean Err: 14.61px | <=1px: 28.1% | <=2px: 61.9%
Epoch [04/10] (575.2s) | Loss: 0.3706 | Mean Err: 12.99px | <=1px: 38.8% | <=2px: 74.4%
Epoch [05/10] (578.7s) | Loss: 0.2120 | Mean Err: 15.47px | <=1px: 51.2% | <=2px: 78.1%
Epoch [06/10] (584.0s) | Loss: 0.0555 | Mean Err: 10.41px | <=1px: 60.0% | <=2px: 83.1%
Epoch [07/10] (577.9s) | Loss: 0.0104 | Mean Err: 11.47px | <=1px: 72.5% | <=2px: 87.5%
Epoch [08/10] (580.0s) | Loss: 0.0050 | Mean Err: 9.34px | <=1px: 76.9% | <=2px: 90.0%
Epoch [09/10] (576.8s) | Loss: 0.0033 | Mean Err: 8.30px | <=1px: 81.2% | <=2px: 88.8%
Epoch [10/10] (577.6s) | Loss: 0.0024 | Mean Err: 9.68px | <=1px: 80.6% | <=2px: 87.5%
[+] Model checkpoint exported: '/kaggle/working/weights/driftse